<a href="https://colab.research.google.com/github/KuldeepIsharwal/Machine-Learning-Basics/blob/main/function_transformer(mathematical).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

fEATURE eng.
*   feature transform
      *   mathematical transformations(log,   reciprocal,power)








Another part of feature eng.
 sometimes we need a custom transformation — a log transform, a custom cleaning function, a domain-specific calculation. FunctionTransformer lets us write any function and treat it as a proper sklearn transformer.


 algos like regression and L reg assumes that data is normal
 but dts and rf dont care abt it

The whole idea is to try to convert the distribution(pdf) in 'NORMAL' so that prediction is easy

sklearn


*   function transform


       *   log
       *   sq/sq root
       *    reciprocal or any custom

*   power transform

      *   box-cox
      *   yeo jhonson





How to find id data is normal ?

call sns.distplot
call pandas.skew()


Log Transformation ::

used on data that is right skewed uk the reason(see log graph)

Reciprocal transform ::

1/x

sq trans ::

x^2 on right skewed

In [ ]:
import pandas as pd
import numpy as np

import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer


In [ ]:
df = pd.read_csv('/content/train.csv',usecols=['Age','Fare','Survived'])

In [ ]:
df['Age'].fillna(df['Age'].mean(),inplace=True)

In [ ]:
x = df.iloc[:,1:3]
y = df.iloc[:,0]

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
plt.figure(figsize=(13,5))
plt.subplot(1,2,1)
sns.distplot(x_train['Age'], kde=True)

plt.subplot(1,2,2)
stats.probplot(x_train['Age'],dist="norm",plot=plt)
plt.title('QQ')
plt.show()

In [ ]:
plt.figure(figsize=(13,5))
plt.subplot(1,2,1)
sns.distplot(x_train['Fare'], kde=True)

plt.subplot(1,2,2)
stats.probplot(x_train['Fare'],dist="norm",plot=plt)
plt.title('QQ')
plt.show()

In [ ]:
clf = LogisticRegression()
clf.fit(x_train,y_train)
y_pred = clf.predict(x_test)
print(accuracy_score(y_test,y_pred))

In [ ]:
clf2= DecisionTreeClassifier()
clf2.fit(x_train,y_train)
y_pred = clf2.predict(x_test)
print(accuracy_score(y_test,y_pred))

now lets see change if we apply log transform

In [ ]:
trf = FunctionTransformer(np.log1p)
x_train_transformed = trf.transform(x_train)
x_test_transformed = trf.transform(x_test)

In [ ]:
clf = LogisticRegression()
clf.fit(x_train_transformed,y_train)
y_pred = clf.predict(x_test_transformed)
print(accuracy_score(y_test,y_pred))

In [ ]:
clf2= DecisionTreeClassifier()
clf2.fit(x_train_transformed,y_train)
y_pred = clf2.predict(x_test_transformed)
print(accuracy_score(y_test,y_pred))

In [ ]:
x_transformed = trf.fit_transform(x)
clf = LogisticRegression()
clf2= DecisionTreeClassifier()
print(cross_val_score(clf,x_transformed,y,cv=10,scoring='accuracy').mean())
print(cross_val_score(clf2,x_transformed,y,cv=10,scoring='accuracy').mean())

In [ ]:
def reciprocal(x):
  return 1/(x+0.0001)

In [ ]:
trf4 = FunctionTransformer(reciprocal)
x_transformed = trf4.fit_transform(x)

In [ ]:
cld = LogisticRegression()
x_train_transformed,x_test_transformed,y_train,y_test = train_test_split(x_transformed,y,test_size=0.2,random_state=42)
cld.fit(x_train_transformed, y_train)
y_pred = cld.predict(x_test_transformed)
print(accuracy_score(y_test,y_pred))

poor accuracy now

lesson :: apply various transformations and find out which one fits ur data best

#Power Transformations ::



*   box-cox ✈ It tries different values of λ (lambda) and picks the one that makes the data most normal.
The formula:
if λ ≠ 0:   x_new = (x^λ - 1) / λ
if λ = 0:   x_new = log(x)

λ range = [-5,5]

works strictly on x > 0

*   yeo-jhonson ✈ Same idea — finds optimal λ — but uses a modified formula that handles zero and negative values:
if x ≥ 0 and λ ≠ 0:  x_new = ((x+1)^λ - 1) / λ
if x ≥ 0 and λ = 0:  x_new = log(x+1)
if x < 0 and λ ≠ 2:  x_new = -((-x+1)^(2-λ) - 1) / (2-λ)
if x < 0 and λ = 2:  x_new = -log(-x+1)




In [ ]:
from sklearn.metrics import r2_score
from sklearn.preprocessing import PowerTransformer

In [ ]:
df2 = pd.read_csv('/content/insurance.csv')

In [ ]:
df2.head()

In [ ]:
df2.isnull().sum()

In [ ]:
df.shape

In [ ]:
df2.describe()

In [ ]:
x = df2.drop('bmi',axis=1)
y = df2['bmi']

In [ ]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Identify categorical columns
categorical_cols = x_train.select_dtypes(include=['object', 'category']).columns

# Create a ColumnTransformer to apply OneHotEncoder to categorical columns
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough' # Keep other columns as they are
)

# Apply preprocessing to x_train and x_test
x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

lr = LinearRegression()
lr.fit(x_train_processed,y_train)
y_pred = lr.predict(x_test_processed)
print(r2_score(y_test,y_pred))

In [ ]:
for col in x_train.select_dtypes(include=np.number).columns:
  plt.figure(figsize=(13,5))
  plt.subplot(1,2,1)
  sns.histplot(x_train[col], kde=True) # Changed distplot to histplot as per earlier deprecation warning fix

  plt.subplot(1,2,2)
  stats.probplot(x_train[col],dist="norm",plot=plt)
  plt.title('QQ')
  plt.show()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer, OneHotEncoder

# Identify numerical and categorical columns
numerical_cols = x_train.select_dtypes(include=np.number).columns
categorical_cols = x_train.select_dtypes(include=['object', 'category']).columns

# Create a ColumnTransformer to apply different transformations to different columns
# For Box-Cox, we apply it to numerical columns, and OneHotEncoder to categorical
# Box-Cox requires positive values, so we add a small constant to ensure this.
# Using StandardScaler before PowerTransformer can sometimes help stabilize, but PowerTransformer handles scaling internally.
preprocessor_boxcox = ColumnTransformer(
    transformers=[
        ('num', PowerTransformer(method='box-cox'), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough' # Keep other columns (if any) as they are
)

# Apply preprocessing to x_train and x_test
x_train_transformed_boxcox = preprocessor_boxcox.fit_transform(x_train + 0.000001) # Add small constant to ensure positivity for Box-Cox
x_test_transformed_boxcox = preprocessor_boxcox.transform(x_test + 0.000001)

# Retrieve lambdas for numerical columns (if needed, but PowerTransformer's internal lambdas might not map directly to original columns due to ColumnTransformer)
# This part can be tricky because ColumnTransformer outputs a single array and PowerTransformer stores lambdas in the order of transformed columns.
# For simplicity, we'll just demonstrate the R2 score with the full transformation.
print("Shape of x_train_transformed_boxcox:", x_train_transformed_boxcox.shape)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

lr_boxcox = LinearRegression()
lr_boxcox.fit(x_train_transformed_boxcox, y_train)
y_pred_boxcox = lr_boxcox.predict(x_test_transformed_boxcox)
print("R-squared with Box-Cox transformation:", r2_score(y_test, y_pred_boxcox))

In [ ]:
#yeo jhonson
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# Ensure x, y are correctly defined from df2 for the insurance dataset context
x = df2.drop('bmi',axis=1)
y = df2['bmi']

# Re-perform the train-test split to ensure x_train, x_test, y_train, y_test are up-to-date
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

# Identify numerical and categorical columns from the correctly loaded x_train
numerical_cols = x_train.select_dtypes(include=np.number).columns
categorical_cols = x_train.select_dtypes(include=['object', 'category']).columns

# Create a ColumnTransformer to apply different transformations for Yeo-Johnson
preprocessor_yeojohnson = ColumnTransformer(
    transformers=[
        ('num', PowerTransformer(method='yeo-johnson'), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough' # Keep other columns (if any) as they are
)

# Apply preprocessing to x_train and x_test
x_train_transformed_yeojohnson = preprocessor_yeojohnson.fit_transform(x_train)
x_test_transformed_yeojohnson = preprocessor_yeojohnson.transform(x_test)

print("Shape of x_train_transformed_yeojohnson:", x_train_transformed_yeojohnson.shape)

# Train and evaluate Linear Regression model with Yeo-Johnson transformed data
lr_yeojohnson = LinearRegression()
lr_yeojohnson.fit(x_train_transformed_yeojohnson, y_train)
y_pred_yeojohnson = lr_yeojohnson.predict(x_test_transformed_yeojohnson)
print("R-squared with Yeo-Johnson transformation:", r2_score(y_test, y_pred_yeojohnson))